# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irssaa29/Machine-learning_01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** ML Appendix — Random Forest Feature Importance for Health Score (page 27)
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of Health Score, and to its credit, explicitly discloses: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."

**My methodology question:** Health Score is defined as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts) — so two of the three top "predictors" (Position, Impressions) are literally components used to compute the label itself. This is the same structural pattern I found in my own Week 5 model, where h1_ctr accounted for 99.9% of feature importance because it was mathematically tied to how my label was defined. I'd ask: given this overlap, would it be clearer to report feature importance only for inputs that are not part of the Health Score formula (like Content Age, WordCount, Days Visible — all showing 0% importance here), since those are the only ones that could reveal genuinely new information rather than the model rediscovering its own label's arithmetic? The paper's honesty about this is good practice — I'm asking whether the chart itself could be restructured to make the finding less misleading at a glance, since a reader skimming the bar chart might miss the caveat in the text.
**Finding 2:** ML Appendix — Growth Prediction via Logistic Regression (page 29)
The paper reports 71% holdout accuracy predicting growing vs. declining pages, with Content Age as the strongest negative signal and Days Visible / recent impressions as the strongest positive signals. trend_direction (growing/declining) is defined from 30-day-vs-previous-30-day impression change (per the "Understanding the Metrics" section).

**My methodology question:** the methodology section mentions an 80/20 split for this model but doesn't specify whether it's a random split or grouped by brand (there are 57 brands in the portfolio). In my own Week 5/6 work, I found that a random split let my model partly learn client-specific quirks rather than general patterns, and a client-grouped split gave a more honest, lower number. I'd ask: was this 80/20 split random or brand-grouped? If random, some brands' pages might appear in both train and test, which could inflate the reported 71% relative to how the model would perform on a brand it's never seen — which matters a lot if a reader is trying to judge whether this pattern would hold for their own site.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/irssaa29/Machine-learning_01"
REPO_DIR = "Machine-learning_01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "scikit-learn"], check=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Rebuild df_pair ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
)
df_march["report_date"] = pd.to_datetime(df_march["report_date"])
first_half = df_march[df_march["report_date"].dt.day <= 15]
second_half = df_march[df_march["report_date"].dt.day > 15]

feat = first_half.groupby(["content_hash_id", "client_hash_id"]).agg(
    h1_impressions=("gsc_impressions", "sum"),
    h1_avg_position=("gsc_avg_position", "mean"),
    h1_clicks=("gsc_clicks", "sum"),
    h1_active_days=("report_date", "nunique"),
).reset_index()
feat["h1_ctr"] = feat["h1_clicks"] / feat["h1_impressions"].replace(0, pd.NA)

h2 = second_half.groupby("content_hash_id").agg(h2_clicks=("gsc_clicks", "sum")).reset_index()
df_pair = feat.merge(h2, on="content_hash_id", how="inner")
df_pair["is_declining"] = (df_pair["h2_clicks"] < df_pair["h1_clicks"]).astype(int)
df_pair = df_pair[(df_pair["h1_avg_position"].notna()) & (df_pair["h1_avg_position"] > 0)].copy()

features = ["h1_impressions", "h1_avg_position", "h1_clicks", "h1_ctr", "h1_active_days"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# --- Client-grouped split ---
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_pair, groups=df_pair["client_hash_id"]))
df_train = df_pair.iloc[train_idx].copy()
df_test = df_pair.iloc[test_idx].copy()

X_train = df_train[features].fillna(0)
y_train = df_train["is_declining"].values
X_test = df_test[features].fillna(0)
y_test = df_test["is_declining"].values

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

# --- Random split ---
X_all = df_pair[features].fillna(0)
y_all = df_pair["is_declining"].values

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

tree_rand = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_rand.fit(X_train_rand, y_train_rand)
scores_rand = tree_rand.predict_proba(X_test_rand)[:, 1]

print("=== RANDOM SPLIT (no grouping) ===")
print(f"Base rate: {y_test_rand.mean():.3f}")
for k in (20, 50):
    print(f"Precision@{k}: {precision_at_k(scores_rand, y_test_rand, k):.3f}")

print("\n=== GROUPED SPLIT (by client_hash_id) ===")
print(f"Base rate: {y_test.mean():.3f}")
for k in (20, 50):
    print(f"Precision@{k}: {precision_at_k(tree_scores, y_test, k):.3f}")

/tmp/ipykernel_2023/1654561450.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = df_train[features].fillna(0)
/tmp/ipykernel_2023/1654561450.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = df_test[features].fillna(0)
/tmp/ipykernel_2023/1654561450.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_all = df_p

=== RANDOM SPLIT (no grouping) ===
Base rate: 0.192
Precision@20: 0.550
Precision@50: 0.640

=== GROUPED SPLIT (by client_hash_id) ===
Base rate: 0.233
Precision@20: 0.700
Precision@50: 0.800


**Random split vs. grouped split (before/after):** Precision@20 was 0.550 (random) vs. 0.700 (grouped); Precision@50 was 0.640 (random) vs. 0.800 (grouped) — base rates 0.192 and 0.233 respectively.

This is the opposite of the typical pattern (where random splits usually look inflated due to memorization). With only 44 unique clients — and just 14 landing in the grouped test set — the grouped split has high variance: which specific clients happen to fall into the test set can swing the score substantially, independent of memorization. This is a real, known limitation of grouped splits with a small number of groups: it removes bias from client memorization, but introduces new variance from small-sample group selection. I would not treat either number as fully trustworthy on its own; a more robust check would run grouped k-fold cross-validation across multiple client splits and report the spread, not just one split.Separately, both numbers here still include the structural h1_ctr == 0 artifact identified in Week 5 (pages that cannot mathematically decline), so neither number should be read as a clean estimate of real model skill — see Section 3's leakage audit below for the corrected, artifact-free comparison.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from the skill file against my final feature set (h1_impressions, h1_avg_position, h1_clicks, h1_ctr, h1_active_days) and label (is_declining):

In [6]:
# --- Attack check 1: deliberately add a label-derived feature, confirm test harness catches it ---
df_train_leak = df_train.copy()
df_test_leak = df_test.copy()
df_train_leak["clicks_change"] = df_train_leak["h2_clicks"] - df_train_leak["h1_clicks"]
df_test_leak["clicks_change"] = df_test_leak["h2_clicks"] - df_test_leak["h1_clicks"]

features_leaky = features + ["clicks_change"]
X_train_leaky = df_train_leak[features_leaky].fillna(0)
X_test_leaky = df_test_leak[features_leaky].fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_leaky.fit(X_train_leaky, y_train)
scores_leaky = tree_leaky.predict_proba(X_test_leaky)[:, 1]

print("WITHOUT suspect feature — Precision@50:", round(precision_at_k(tree_scores, y_test, 50), 3))
print("WITH suspect feature (clicks_change)  — Precision@50:", round(precision_at_k(scores_leaky, y_test, 50), 3))

# --- Attack check 2: structural artifact re-check from Week 5 (h1_ctr == 0 floor) ---
zero_ctr_test = df_test[df_test["h1_ctr"] == 0]
print("\nTest pages with h1_ctr == 0:", len(zero_ctr_test))
print("Declining rate among them (should be 0.0 if the floor holds):", zero_ctr_test["is_declining"].mean())

WITHOUT suspect feature — Precision@50: 0.8
WITH suspect feature (clicks_change)  — Precision@50: 1.0

Test pages with h1_ctr == 0: 24756
Declining rate among them (should be 0.0 if the floor holds): 0.0


/tmp/ipykernel_2023/2972966521.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_leaky = df_train_leak[features_leaky].fillna(0)
/tmp/ipykernel_2023/2972966521.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_leaky = df_test_leak[features_leaky].fillna(0)


Timeline drawn: all five features come strictly from March 1–15 ("before"); the label compares to March 16–31 ("after"). No overlap.
 No label-derived or sibling columns in features: confirmed via the deliberate-leak test above — adding clicks_change (h2_clicks − h1_clicks, directly derived from the label's own definition) causes Precision@50 to jump toward 1.0, confirming my test harness correctly detects leakage when it's present. None of my five real features are label-derived.
 No product flags used: the warehouse table (fact_content_daily_performance) contains no FlyRank optimization flags or composite scores — only raw GSC/GA4 metrics.
 Population selection checked: I restricted to pages with h1_avg_position > 0 (real position data) — this is a pre-decision-window filter, not an outcome-window one, so it doesn't leak future information. Documented as a scoping choice in Week 3/4.
 Split grouped by client: confirmed via GroupShuffleSplit on client_hash_id, zero client overlap verified in Section 2.
 Base rate printed next to every metric: done throughout Sections 2–3.
 Top feature importance sanity-checked: flagged in Week 5 — h1_ctr at 99.9% importance was investigated and found to be a structural artifact (pages with h1_ctr == 0 always have h1_clicks == 0, and can never mathematically decline under this label — confirmed again above). After restricting to h1_ctr > 0, importance spread across four real signals, which I consider the trustworthy result.
 Metrics recomputed out-of-fold: all Section 2/3 numbers are computed on held-out df_test, never on training data.Sealed/holdout receipts committed: not yet formally sealed — this notebook's df_test is a fresh 30% held out each run, not a permanently frozen sealed set. Worth doing for the final capstone model, not required this week.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest original sentence (from Week 5):** "Precision@20 = 0.90 (vs. a 0.583 base rate) reflects genuine, explainable model skill."

**Rewritten in safe, public-appropriate language:** "In this observational sample, restricting to pages with any first-half click activity, the decision tree's top-20 ranking showed a directional lift over the base rate (0.90 vs. 0.583) on a held-out set of clients not seen during training. This is a measured, decision-support result from a single grouped train/test split, not a proven, generalizable causal effect — the small number of unique clients (44 total, 14 in the held-out test) means this specific number carries real sampling variance, as shown by the gap between my random-split and grouped-split results in Section 2. I would treat this as a promising directional signal worth further validation (e.g. repeated grouped cross-validation) rather than a finalized, production-ready claim." **What changed and why:** the original sentence used "genuine" and "reflects... skill" — language that implies a settled, confident causal-adjacent claim. The rewrite adds: (1) "observational sample" and "directional" instead of implying proof, (2) explicit acknowledgment of the split-variance problem found in Section 2, (3) "measured, decision-support" language per this week's standard, and (4) an honest scope limitation (single split, small client count) rather than presenting one run's number as a stable truth.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.